# 06 — Dựng dataset
Ghép cấu trúc, mật độ, năng lượng PBE và nhãn tham chiếu; lưu HDF5 cùng manifest và metadata. Mẫu không hội tụ, lệch ∫ρ, thiếu nhãn hoặc chứa NaN bị loại và ghi lý do vào manifest.

**Đầu ra:** `data/ml_xc_dataset.h5`, `data/processed/dataset_manifest.csv`, `dataset_metadata.json`.

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf pymatgen h5py

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd, time, platform
from utils import data_utils as dtu

H5 = f"{DATA}/ml_xc_dataset.h5"
sel = pd.read_csv(f"{PROC}/selected.csv")
labels = pd.read_csv(f"{DATA}/reference_labels/labels.csv")
settings = json.load(open(f"{PROC}/pbe_settings.json"))["settings"]
method = json.load(open(f"{DATA}/reference_labels/labels_method.json"))
pre = json.load(open(f"{PROC}/preprocess_report.json"))

In [ ]:
import pyscf, pymatgen
meta = {"created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "mp_database_version": sorted(sel["mp_database_version"].dropna().unique().tolist()),
        "pyscf": pyscf.__version__, "python": platform.python_version(),
        "preprocess": pre.get("settings"), "reference_labels": method}
manifest = dtu.build_h5(H5, sel, f"{DATA}/densities", labels, settings, meta)
manifest = manifest.merge(sel[["material_id", "formula_pretty", "n_atoms", "band_gap_eV", "run_type"]], on="material_id")
manifest.to_csv(f"{PROC}/dataset_manifest.csv", index=False)
json.dump(meta | {"pbe_settings": settings}, open(f"{PROC}/dataset_metadata.json", "w"),
          indent=2, ensure_ascii=False, default=str)
print(f"Đã lưu {H5}  ({os.path.getsize(H5)/1e6:.1f} MB)")
print("Đưa vào dataset:", int(manifest.included.sum()), "/", len(manifest))
bad = manifest[~manifest.included]
print("\nLý do loại:", bad.reason.value_counts().to_dict() if len(bad) else "không có")
print("\nSố mẫu theo split:", manifest[manifest.included].split.value_counts().to_dict())

## Kiểm tra lại file vừa ghi

In [ ]:
t = dtu.read_targets(H5)
assert len(t) == int(manifest.included.sum())
assert t["delta_ref_pbe_eV_per_atom"].notna().all()
one = dtu.load_sample(H5, t.material_id.iloc[0])
print("Ví dụ:", one["material_id"], one["formula"], "| mesh", one["rho"].shape, "| Δ_ref-PBE =",
      round(one["delta_ref_pbe_Ha"] * 27.2114 / one["n_atoms"] * 1e3, 2), "meV/nguyên tử")
t.groupby("split")[["delta_ref_pbe_eV_per_atom", "delta_lda_pbe_eV_per_atom"]].agg(["mean", "std", "count"]).round(4)